In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
    .appName("CovidData") \
    .getOrCreate()

In [3]:
df = spark.read.csv("covid-data.csv", header=True, inferSchema=True)
df.createOrReplaceTempView("covid_data")

In [4]:
countries_top15_percentage = spark.sql("""
    SELECT iso_code,
        location AS `страна`,
        ROUND(total_cases / population * 100, 2) AS `процент переболевших`
    FROM covid_data
    WHERE date = "2021-03-31"
        AND iso_code NOT LIKE 'OWID_%'
        AND population IS NOT NULL
    ORDER BY `процент переболевших` DESC
    LIMIT 15
""")
countries_top15_percentage.show()

+--------+-------------+--------------------+
|iso_code|       страна|процент переболевших|
+--------+-------------+--------------------+
|     AND|      Andorra|               15.54|
|     MNE|   Montenegro|               14.52|
|     CZE|      Czechia|               14.31|
|     SMR|   San Marino|               13.94|
|     SVN|     Slovenia|               10.37|
|     LUX|   Luxembourg|                9.85|
|     ISR|       Israel|                9.63|
|     USA|United States|                 9.2|
|     SRB|       Serbia|                8.83|
|     BHR|      Bahrain|                8.49|
|     PAN|       Panama|                8.23|
|     PRT|     Portugal|                8.06|
|     EST|      Estonia|                8.02|
|     SWE|       Sweden|                7.97|
|     LTU|    Lithuania|                7.94|
+--------+-------------+--------------------+



In [5]:
countries_top10_LastWeekMarch2021 = spark.sql("""
    SELECT date AS `число`,
        location AS `страна`,
        CAST(new_cases AS INT) AS `кол-во новых случаев`
    FROM covid_data
    WHERE date BETWEEN "2021-03-25" AND "2021-03-31"
        AND iso_code NOT LIKE 'OWID_%'
    ORDER BY `кол-во новых случаев` DESC
    LIMIT 10
""")
countries_top10_LastWeekMarch2021.show()

+----------+-------------+--------------------+
|     число|       страна|кол-во новых случаев|
+----------+-------------+--------------------+
|2021-03-25|       Brazil|              100158|
|2021-03-31|       Brazil|               90638|
|2021-03-27|       Brazil|               85948|
|2021-03-30|       Brazil|               84494|
|2021-03-26|       Brazil|               84245|
|2021-03-26|United States|               77321|
|2021-03-31|        India|               72330|
|2021-03-29|United States|               69429|
|2021-03-28|        India|               68020|
|2021-03-25|United States|               67465|
+----------+-------------+--------------------+



In [6]:
Russia_delta_LastWeekMarch2021 = spark.sql("""
    WITH Russia AS (
        SELECT to_date(date) AS date,
            CAST(MAX(new_cases) AS INT) AS new_cases
        FROM covid_data
        WHERE location = 'Russia'
            AND to_date(date) BETWEEN "2021-03-24" AND "2021-03-31"
        GROUP BY to_date(date)
    ),
    calculation AS (
        SELECT date AS `число`,
            LAG(new_cases) OVER (ORDER BY date) AS `кол-во новых случаев вчера`,
            new_cases AS `кол-во новых случаев сегодня`,
            new_cases - LAG(new_cases) OVER (ORDER BY date) AS `дельта`
        FROM Russia
    )    
    SELECT *
    FROM calculation
    WHERE `число` >= '2021-03-25'
    ORDER BY `число`;
""")
Russia_delta_LastWeekMarch2021.show()

+----------+--------------------------+----------------------------+------+
|     число|кол-во новых случаев вчера|кол-во новых случаев сегодня|дельта|
+----------+--------------------------+----------------------------+------+
|2021-03-25|                      8769|                        9128|   359|
|2021-03-26|                      9128|                        9073|   -55|
|2021-03-27|                      9073|                        8783|  -290|
|2021-03-28|                      8783|                        8979|   196|
|2021-03-29|                      8979|                        8589|  -390|
|2021-03-30|                      8589|                        8162|  -427|
|2021-03-31|                      8162|                        8156|    -6|
+----------+--------------------------+----------------------------+------+

